# Capítulo 2: Um Curso Rápido de Python

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 2 de Grus (2019).

> As pessoas continuam malucas por Python depois de vinte e cinco anos, o que me custa acreditar.
>
> — Michael Palin

> **❗ Importante — Este capítulo assume que você já programa**
>
> Você chegou aqui em Ciência da Computação: já escreveu código, já sabe o que é uma função, um laço e uma condicional. O básico está aqui, mas como referência rápida — uma passada pelo que muda de sintaxe em relação ao que você já conhece.
>
> A profundidade vai para o que o código do Grus (2019) realmente exige e que provavelmente é novo para você:
>
> - `assert` como documentação executável
> - anotações de tipo
> - `zip`, `lambda` e funções de primeira classe
> - `Counter` e `defaultdict`
> - `enumerate` e `NamedTuple`
> - geradores (`yield`)
>
> O primeiro item é o que mais pesa. Os `assert` que salpicam o código do livro-texto não são testes espalhados: **é assim que aquele código afirma o que cada função faz** — e quem não reconhece o padrão lê o livro inteiro achando que são testes fora de lugar.
>
> Cada seção deste capítulo abre dizendo o que assume conhecido e onde vale desacelerar.

Se em algum momento você quiser a versão completa e do zero, a seção de leituras adicionais aponta para ela.

Ao final deste capítulo, você será capaz de:

- Preparar um ambiente Python isolado e explicar por que isso importa num projeto com dependências travadas
- Ler e escrever o idioma do livro-texto: compreensões, `zip`, `enumerate`, funções de primeira classe
- Usar `Counter` e `defaultdict` nas situações em que eles substituem dez linhas de contagem manual
- Empregar `assert` como documentação executável, e distinguir os dois papéis que ele tem no código do livro
- Ler e escrever anotações de tipo, e explicar o que elas fazem e o que não fazem
- Reconhecer geradores e avaliação preguiçosa, e saber quando isso muda o custo de um cálculo

## Seções

| Seção | Tópico |
|---|---|
| [2.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/01-ambiente-e-sintaxe.html) | Ambiente e Sintaxe |
| [2.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) | Funções, Strings e Exceções |
| [2.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/03-estruturas-de-dados.html) | Estruturas de Dados |
| [2.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/04-controle-de-fluxo.html) | Controle de Fluxo |
| [2.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/05-testes-classes-e-geradores.html) | Testes, Classes e Geradores |
| [2.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) | Ferramentas e Anotações de Tipo |

## Ambiente e Sintaxe

> **📌 Nota**
>
> Esta seção corresponde a *The Zen of Python*, *Getting Python*, *Virtual Environments*, *Whitespace Formatting* e *Modules*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você sabe o que é um interpretador, um módulo e uma dependência. O que provavelmente é novo: a disciplina de ambiente isolado e as formas de `import` que este livro usa.

### O Zen do Python

Python tem uma descrição zen-budista dos seus princípios de projeto, que você lê digitando `import this` no interpretador. Um dos mais discutidos é:

> Deve haver um — e preferencialmente apenas um — modo óbvio de fazer.

Código escrito nesse modo óbvio (que pode não ser óbvio de cara para quem chega de outra linguagem) costuma ser chamado de "Pythônico". Este livro tenta escrever Python pythônico, e a palavra vai reaparecer nas próximas seções sempre que houver duas formas de fazer a mesma coisa e uma delas for a preferida.

### Ambientes virtuais

Aqui vale desacelerar, porque é um hábito de engenharia que não depende de saber Python.

O problema: projetos diferentes precisam de versões diferentes das mesmas bibliotecas. Se você instalar tudo no Python do sistema, um projeto que precisa de uma versão nova quebra o outro que precisa da antiga — e você descobre isso no pior momento possível.

A solução é um **ambiente virtual**: uma instalação de Python isolada, por projeto, com as suas próprias bibliotecas.

> **❗ Importante — Isolar não basta: trave as versões**
>
> O ambiente isolado resolve o conflito entre projetos. Falta a outra metade: registrar num arquivo de lock **quais** versões o projeto usa, para que a mesma instalação possa ser reconstruída depois — na sua máquina daqui a um ano, ou na máquina de outra pessoa hoje.
>
> O motivo é concreto. Uma atualização de biblioteca pode mudar em silêncio o comportamento de uma operação: já aconteceu com uma comparação do pandas, que passou a devolver a resposta errada sem erro e sem aviso. Sem as versões travadas, quando um resultado muda você não tem como saber se mudou porque o dado mudou, porque você mexeu no código, ou porque uma dependência foi atualizada embaixo de você.

### Formatação por espaço em branco

Python usa **indentação** para delimitar blocos, onde outras linguagens usam chaves:

In [ ]:
for i in [1, 2, 3, 4, 5]:
    for j in [1, 2, 3, 4, 5]:
        print(i + j, end=" ")   # a indentação diz o que está dentro de quê
    print("|", end=" ")

Isso torna o código legível e cria uma classe de erro que outras linguagens não têm: **a indentação errada muda o significado sem gerar erro de sintaxe**. Um bloco recuado um nível a mais ou a menos ainda é código válido, com comportamento diferente.

O espaço em branco é ignorado dentro de parênteses e colchetes, o que permite quebrar expressões longas:

In [ ]:
lista_longa = [1, 2, 3,
               4, 5, 6,
               7, 8, 9]

soma = (1 + 2 + 3 +
        4 + 5 + 6)

lista_longa, soma

Você vai ver esse recurso o tempo todo no código do livro — as definições de função com muitos parâmetros são quebradas assim, um por linha.

### Módulos e imports

Recursos que não são carregados por padrão precisam ser importados. A forma mais simples traz o módulo inteiro:

In [ ]:
import re

meu_regex = re.compile("[0-9]+", re.I)
type(meu_regex).__name__

Se você já tiver um `re` no seu código, pode dar um apelido:

In [ ]:
import re as regex

regex.compile("[0-9]+") is not None

Apelidos também servem quando o nome do módulo é comprido ou quando existe uma convenção — `import matplotlib.pyplot as plt` é a que você mais vai encontrar neste livro.

E dá para importar valores específicos de um módulo, usando-os diretamente:

In [ ]:
from collections import defaultdict, Counter

contador = Counter()
type(contador).__name__

> **⚠️ Atenção — Uma quarta forma, que você não deve usar**
>
> Existe ainda uma forma que importa tudo para o espaço de nomes atual:
>
> ```python
> from re import *      # nunca faça isto
> ```
>
> Ela sobrescreve silenciosamente qualquer nome que você já tivesse. O módulo `re` tem uma função chamada `match`; se você tinha uma variável `match`, ela desapareceu — e você vai descobrir isso muito depois, com um erro que não aponta para a causa.
>
> Use `import módulo` ou `from módulo import nome`, listando sempre o que você traz.

## Funções, Strings e Exceções

> **📌 Nota**
>
> Esta seção corresponde a *Functions*, *Strings* e *Exceptions*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você sabe o que é uma função, um parâmetro e uma exceção. O que provavelmente é novo: funções como valores de primeira classe, `lambda`, argumentos nomeados com padrão, e o estilo pythônico de tratar erro.

### Funções são valores

Uma função em Python se define com `def`:

In [ ]:
def double(x):
    """
    Aqui vai a docstring, opcional, que explica o que a função faz.
    Por exemplo, esta função multiplica sua entrada por 2.
    """
    return x * 2

double(21)

O que importa aqui não é a sintaxe, e sim que **funções em Python são objetos de primeira classe**. Dá para atribuí-las a variáveis e passá-las como argumento para outras funções:

In [ ]:
def apply_to_one(f):
    """Chama a função f com 1 como argumento"""
    return f(1)

my_double = double            # atribui a função a uma variável
x = apply_to_one(my_double)   # e passa a variável adiante

assert x == 2
x

Também dá para criar funções curtas e anônimas com `lambda`:

In [ ]:
y = apply_to_one(lambda x: x + 4)

assert y == 5
y

> **❗ Importante — Onde `lambda` aparece**
>
> No código do livro, quase sempre nos mesmos dois lugares: como argumento `key` de uma ordenação, ou para gerar os elementos de uma matriz.
>
> ```python
> num_friends_by_id.sort(key=lambda id_and_friends: id_and_friends[1])
> identity_matrix = make_matrix(n, n, lambda i, j: 1 if i == j else 0)
> ```
>
> Você não precisa gostar de `lambda`. Precisa reconhecê-lo, porque o padrão "passar um pedacinho de comportamento para outra função" atravessa o livro.

Parâmetros podem ter valores padrão, e argumentos podem ser passados pelo nome:

In [ ]:
def full_name(first="What's-his-name", last="Something"):
    return first + " " + last

assert full_name("Joel", "Grus")   == "Joel Grus"
assert full_name("Joel")           == "Joel Something"
assert full_name(last="Grus")      == "What's-his-name Grus"

full_name(last="Grus")

A terceira linha é o ponto: `full_name(last="Grus")` pula o primeiro parâmetro pelo nome. Isso aparece bastante em chamadas com muitos parâmetros opcionais — as de matplotlib, no [Capítulo 3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap03/index.html), são quase todas assim.

### Strings

Aspas simples ou duplas, indiferente. A barra invertida escapa caracteres especiais:

In [ ]:
tab_string = "\t"        # representa o caractere de tabulação
assert len(tab_string) == 1

not_tab_string = r"\t"   # string "crua": a barra e o t, literalmente
assert len(not_tab_string) == 2

len(tab_string), len(not_tab_string)

A **string crua** (`r"..."`) importa mais do que parece: é o que se usa para expressões regulares, onde a barra invertida tem significado próprio e escapá-la duas vezes fica ilegível.

Para montar strings a partir de valores, use **f-strings**:

In [ ]:
first_name = "Joel"
last_name = "Grus"

f"{first_name} {last_name}"

> **🟩 Exemplo**
>
> Existem formas mais antigas de fazer isso — concatenação com `+`, o operador `%`, o método `.format()`. Você vai encontrá-las em código antigo, e elas funcionam.
>
> A f-string é a forma óbvia hoje, no sentido do Zen do Python: uma coisa, um jeito.

### Exceções

Quando algo dá errado, Python levanta uma exceção. Sem tratamento, o programa para:

In [ ]:
try:
    print(0 / 0)
except ZeroDivisionError:
    print("não se pode dividir por zero")

> **🔷 Conceito**
>
> Vale notar uma diferença cultural em relação a linguagens onde exceção é reservada para o excepcional.
>
> Em Python, tratar erro com `try`/`except` é comum e idiomático — inclusive em situações previsíveis. O estilo tem até nome: *é mais fácil pedir perdão do que permissão*. Em vez de checar antes se a chave existe no dicionário, muitas vezes se tenta acessar e se trata o `KeyError`.
>
> Isso não significa capturar tudo. Capture a exceção **específica** que você sabe tratar. Um `except:` pelado engole também os erros que você gostaria de ver.

Você vai ver esse padrão no [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html), quando linhas malformadas de um arquivo de dados precisarem ser puladas sem derrubar o processamento inteiro.

## Estruturas de Dados

> **📌 Nota**
>
> Esta seção corresponde a *Lists*, *Tuples*, *Dictionaries*, *defaultdict*, *Counters* e *Sets*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você conhece arranjos, listas, tabelas hash e conjuntos como estruturas. O que muda: a sintaxe de fatiamento e desempacotamento. **O que vale desacelerar: `defaultdict` e `Counter`** — os dois substituem laços de contagem que você provavelmente escreveria à mão, e por isso aparecem em quase todo capítulo daqui em diante.

### Listas e tuplas, rapidamente

Lista é a estrutura ordenada e mutável de Python. O que difere do que você já conhece é o **fatiamento**, que é mais expressivo que o usual:

In [ ]:
x = list(range(10))

primeiros_tres  = x[:3]      # [0, 1, 2]
ultimos_tres    = x[-3:]     # [7, 8, 9]
sem_primeiro_e_ultimo = x[1:-1]
cada_terceiro   = x[::3]     # [0, 3, 6, 9]
de_cinco_a_tres = x[5:2:-1]  # [5, 4, 3]  — passo negativo inverte

primeiros_tres, ultimos_tres, cada_terceiro, de_cinco_a_tres

O operador `in` verifica pertinência — e é **linear** numa lista, o que importa quando a lista é grande:

In [ ]:
1 in [1, 2, 3], 0 in [1, 2, 3]

**Tuplas** são listas imutáveis: quase tudo que se faz com lista se faz com tupla, menos modificar. Elas servem para retornar múltiplos valores e para desempacotamento:

In [ ]:
def sum_and_product(x, y):
    return (x + y), (x * y)

sp = sum_and_product(2, 3)
s, p = sum_and_product(5, 10)

# desempacotamento também troca valores sem variável temporária
x, y = 1, 2
x, y = y, x

sp, s, p, (x, y)

> **📌 Nota**
>
> `x, y = y, x` não é truque: o lado direito é avaliado por inteiro antes da atribuição, então a troca é atômica do ponto de vista de quem lê. Aparece bastante em código numérico.

### Dicionários

Dicionário associa chaves a valores, com busca rápida. A parte que vale atenção é o que acontece quando a chave **não** existe:

In [ ]:
grades = {"Joel": 80, "Tim": 95}

# [] levanta KeyError se a chave não existe
tem_joel = "Joel" in grades
tem_kate = "Kate" in grades

# .get() devolve um padrão em vez de estourar
nota_joel = grades.get("Joel", 0)
nota_kate = grades.get("Kate", 0)     # 0, o padrão
sem_nota  = grades.get("Ninguém")     # None, o padrão do padrão

tem_joel, tem_kate, nota_joel, nota_kate, sem_nota

Dicionários têm `keys()`, `values()` e `items()`, e este último é o que você vai ver o tempo todo:

In [ ]:
for nome, nota in grades.items():
    print(f"{nome}: {nota}")

### `defaultdict`

Aqui vale desacelerar.

Imagine contar palavras de um documento. A versão sem `defaultdict` exige tratar o caso "chave ainda não existe" toda vez:

In [ ]:
document = ["o", "rato", "roeu", "a", "roupa", "do", "rei", "de", "roma",
            "o", "rei", "de", "roma", "roeu", "o", "rato"]

# versão 1: checar antes
word_counts = {}
for word in document:
    if word in word_counts:
        word_counts[word] += 1
    else:
        word_counts[word] = 1

word_counts

Um `defaultdict` faz isso sozinho: quando você acessa uma chave que não existe, ele a cria usando a função que você deu na construção.

In [ ]:
from collections import defaultdict

word_counts = defaultdict(int)   # int() devolve 0
for word in document:
    word_counts[word] += 1       # sem checagem: a chave nasce com 0

dict(word_counts)

O argumento pode ser qualquer função sem parâmetros — e é aí que ele fica realmente útil:

In [ ]:
dd_list = defaultdict(list)      # list() devolve []
dd_list[2].append(1)             # dd_list agora tem {2: [1]}

dd_dict = defaultdict(dict)      # dict() devolve {}
dd_dict["Joel"]["City"] = "Seattle"

dict(dd_list), dict(dd_dict)

> **🟩 Exemplo**
>
> `defaultdict(list)` é o padrão de **agrupar** — para cada chave, acumular uma lista de coisas.
>
> Você já viu esse padrão duas vezes no [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html), sem que ele fosse nomeado: montando o índice de interesses por usuário e o de usuários por interesse, e agrupando salários por faixa de tempo de casa.
>
> Sempre que você for escrever "para cada X, junte os Y correspondentes", é `defaultdict(list)`.

### `Counter`

`Counter` transforma uma sequência em um dicionário de chave para contagem. É o caso de contar palavras, resolvido numa linha:

In [ ]:
from collections import Counter

word_counts = Counter(document)
word_counts

O método que faz `Counter` valer a pena é o `most_common`:

In [ ]:
for word, count in word_counts.most_common(3):
    print(word, count)

> **🔷 Conceito**
>
> `Counter` é um `defaultdict(int)` com contagem automática e ordenação embutida. As três versões da contagem de palavras acima produzem o mesmo resultado — a diferença é quantas decisões você precisou tomar para chegar lá.
>
> No [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html), o `Counter` é o que faz a votação dos k-vizinhos: `Counter(labels).most_common(1)` devolve o rótulo mais votado.

### Conjuntos

`set` guarda elementos distintos, e a razão de usá-lo raramente é a distinção — é a **velocidade**:

In [ ]:
import time

lista_grande = list(range(100_000))
conjunto_grande = set(lista_grande)

alvo = 99_999

t0 = time.perf_counter(); alvo in lista_grande;      t_lista = time.perf_counter() - t0
t0 = time.perf_counter(); alvo in conjunto_grande;   t_conj  = time.perf_counter() - t0

print(f"lista:    {t_lista*1e6:8.1f} µs")
print(f"conjunto: {t_conj*1e6:8.1f} µs")

`in` numa lista **percorre** os elementos até achar o que procura; num conjunto é uma consulta direta, que não depende do tamanho.

Os tempos exatos acima variam a cada execução — é uma medição de relógio, não um cálculo determinístico. O que não varia é a forma: dobre o tamanho da lista e o tempo dela dobra junto; dobre o do conjunto e o tempo fica o mesmo. Com cem mil elementos a diferença já é grande o bastante para aparecer; com dez milhões, a busca linear deixa de ser viável.

É exatamente por isso que, no [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html), o índice de amizades virou um dicionário em vez de continuar sendo uma lista de pares.

## Controle de Fluxo

> **📌 Nota**
>
> Esta seção corresponde a *Control Flow*, *Truthiness*, *Sorting* e *List Comprehensions*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você conhece `if`, `while` e `for`. **O que vale desacelerar: compreensões de lista e ordenação com `key`** — as duas construções que mais mudam a aparência do código do livro em relação ao que você escreveria numa linguagem sem elas.

### Controle de fluxo, rapidamente

`if`, `elif`, `else` funcionam como esperado, e existe uma forma ternária de uma linha:

In [ ]:
x = 7
paridade = "par" if x % 2 == 0 else "ímpar"
paridade

`for` percorre iteráveis diretamente, sem índice:

In [ ]:
for x in range(10):
    if x == 3:
        continue    # vai direto para a próxima iteração
    if x == 5:
        break       # sai do laço
    print(x, end=" ")

### Veracidade

Python tem regras de conversão para booleano que valem conhecer, porque o código do livro se apoia nelas.

São falsos: `False`, `None`, `[]`, `{}`, `""`, `set()`, `0`, `0.0`. Praticamente tudo mais é verdadeiro — o que permite escrever `if lista:` no lugar de `if len(lista) > 0:`.

> **⚠️ Atenção — `is None`, não `== None`**

In [ ]:
x = None

assert x == None, "esta NÃO é a forma pythônica de checar None"
assert x is None, "esta É a forma pythônica de checar None"

x is None

> A diferença importa quando o objeto define o próprio `__eq__` — arrays e DataFrames, por exemplo, comparam elemento a elemento e devolvem um array, não um booleano. `is` compara identidade e nunca é sequestrado.

Duas funções que aparecem bastante:

In [ ]:
all([True, 1, {3}]),   all([True, 1, {}]),   all([]),   any([]), any([True, 1, {}])

Repare no caso vazio: `all([])` é `True` e `any([])` é `False`. É consequência da definição — "todos os elementos satisfazem" é vacuamente verdadeiro quando não há elementos.

### Ordenação com `key`

Toda lista tem `sort()`, que ordena no lugar, e existe `sorted()`, que devolve uma lista nova. Mas a parte que importa é o parâmetro `key`:

In [ ]:
x = [4, 1, 2, 3]

crescente = sorted(x)
decrescente = sorted(x, reverse=True)

# ordena por valor absoluto, do maior para o menor
por_modulo = sorted([-4, 1, -2, 3], key=abs, reverse=True)

crescente, decrescente, por_modulo

`key` recebe uma função que, dado um elemento, devolve o valor pelo qual comparar. É onde os `lambda` do livro moram:

In [ ]:
from collections import Counter

document = ["o", "rato", "roeu", "a", "roupa", "do", "rei", "de", "roma",
            "o", "rei", "de", "roma", "roeu", "o", "rato"]
word_counts = Counter(document)

# ordena as palavras da mais frequente para a menos frequente
wc = sorted(word_counts.items(),
            key=lambda word_and_count: word_and_count[1],
            reverse=True)

wc[:4]

> **🟩 Exemplo**
>
> Esse é exatamente o padrão do [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html), quando ordenamos os usuários da DataSciencester por número de amigos para achar os "conectores".
>
> Ordenar por um critério derivado — em vez de ordenar o próprio valor — é a operação que mais aparece em análise de dados. Vale internalizar a forma.

### Compreensões

Aqui vale desacelerar de verdade. Compreensões são a construção que mais muda a aparência do código.

A ideia: transformar ou filtrar uma coleção em uma expressão só.

In [ ]:
even_numbers = [x for x in range(5) if x % 2 == 0]   # filtrar
squares      = [x * x for x in range(5)]             # transformar
even_squares = [x * x for x in even_numbers]         # as duas coisas

even_numbers, squares, even_squares

Funciona também para dicionários e conjuntos:

In [ ]:
square_dict = {x: x * x for x in range(5)}
square_set  = {x * x for x in [1, -1]}

square_dict, square_set

Quando não se usa o valor do laço, a convenção é chamá-lo de `_`:

In [ ]:
zeros = [0 for _ in even_numbers]   # mesmo comprimento, conteúdo novo
zeros

Uma compreensão pode ter vários `for`, e os posteriores podem usar o resultado dos anteriores:

In [ ]:
pairs = [(x, y)
         for x in range(10)
         for y in range(10)]

increasing_pairs = [(x, y)
                    for x in range(10)
                    for y in range(x + 1, 10)]   # y depende de x

assert len(pairs) == 100
assert len(increasing_pairs) == 9 + 8 + 7 + 6 + 5 + 4 + 3 + 2 + 1
assert all(x < y for x, y in increasing_pairs)

len(pairs), len(increasing_pairs)

> **🔷 Conceito**
>
> Compreensão não é açúcar sintático para deixar o código curto — é uma forma de **dizer o que você quer em vez de como obter**.
>
> Compare: um laço com `append` descreve um procedimento de construção. `[x * x for x in xs if x > 0]` descreve o conjunto resultante. Quando a operação é uma transformação ou um filtro, a segunda forma é mais fácil de ler e mais difícil de errar — não há índice, não há acumulador, não há estado.
>
> Quando a operação **não** é uma transformação simples, o laço continua sendo a escolha certa. Compreensão aninhada de três níveis com condição é ilegível, e o Zen do Python não recomenda isso.

## Testes, Classes e Geradores

> **📌 Nota**
>
> Esta seção corresponde a *Automated Testing and assert*, *Object-Oriented Programming* e *Iterables and Generators*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você conhece classes, herança e testes automatizados como conceitos. **O que vale desacelerar: o `assert` como documentação — o idioma mais frequente do livro inteiro** — e geradores, que mudam o custo de um cálculo sem mudar o que ele calcula.

### `assert` — o idioma central deste livro

Se você levar uma coisa só desta seção, que seja esta.

`assert` verifica uma condição e levanta `AssertionError` se ela for falsa:

In [ ]:
assert 1 + 1 == 2
assert 1 + 1 == 2, "1 + 1 deveria dar 2, mas não deu"

A mensagem opcional depois da vírgula aparece quando a asserção falha. Até aqui, nada de especial.

O que é especial é **como o livro-texto usa isso**. Veja uma função típica do pacote `scratch`:

In [ ]:
from typing import List

def smallest_item(xs: List[float]) -> float:
    return min(xs)

assert smallest_item([10, 20, 5, 40]) == 5
assert smallest_item([1, 0, -1, 2]) == -1

smallest_item([10, 20, 5, 40])

Aquelas duas linhas de `assert` não estão num arquivo de testes. Estão **logo abaixo da função**, no mesmo módulo, e rodam toda vez que o módulo é importado.

> **❗ Importante — Dois papéis diferentes, mesma palavra**
>
> O código do livro usa `assert` de duas maneiras, e vale saber distinguir:
>
> **Como exemplo executável**, depois da função — o caso acima. Ele mostra o que a função faz melhor do que uma frase em prosa, e não pode envelhecer: se alguém quebrar a função, a linha falha na hora do import.
>
> **Como verificação de contrato**, dentro da função:
>
> ```python
> def add(v: Vector, w: Vector) -> Vector:
>     assert len(v) == len(w), "vectors must be the same length"
>     return [v_i + w_i for v_i, w_i in zip(v, w)]
> ```
>
> Aqui o `assert` roda a cada chamada e falha alto quando alguém viola a hipótese da função. Sem ele, o `zip` truncaria em silêncio no menor dos dois vetores e devolveria um resultado mais curto sem aviso — que é pior que um erro.
>
> Os dois papéis atravessam o código do livro de ponta a ponta. Nenhum dos dois é teste espalhado por descuido: é o formato em que o autor afirma o que cada função faz.

> **⚠️ Atenção**
>
> Uma ressalva de engenharia: `assert` é **desligado** quando o Python roda com a flag `-O`. Isso é ótimo para documentação e para pegar erro durante o desenvolvimento, e é ruim para validar entrada de usuário em produção.
>
> Num livro didático, `assert` é a escolha certa. Num servidor que recebe dado de fora, use `if ... raise`.

### Classes, rapidamente

Você já sabe o que é uma classe. O que muda em Python:

In [ ]:
class CountingClicker:
    """Uma classe pode ter docstring, como uma função"""

    def __init__(self, count=0):
        self.count = count

    def __repr__(self):
        return f"CountingClicker(count={self.count})"

    def click(self, num_times=1):
        """Clica o contador algumas vezes"""
        self.count += num_times

    def read(self):
        return self.count

    def reset(self):
        self.count = 0

clicker = CountingClicker()
assert clicker.read() == 0, "clicker deveria começar em 0"
clicker.click()
clicker.click()
assert clicker.read() == 2, "após dois cliques, deveria estar em 2"

clicker

Três pontos específicos de Python: `self` é explícito no primeiro parâmetro de todo método; `__init__` é o construtor; e os métodos com underscores duplos — os *dunder* — implementam comportamento da linguagem, como `__repr__` para a representação em texto.

Herança é direta:

In [ ]:
class NoResetClicker(CountingClicker):
    # Esta classe tem os mesmos métodos de CountingClicker,
    # exceto que o reset não faz nada.
    def reset(self):
        pass

clicker2 = NoResetClicker()
clicker2.click()
clicker2.reset()
assert clicker2.read() == 1, "reset não deveria fazer nada"

clicker2.read()

> **📌 Nota**
>
> Este livro usa pouca orientação a objetos. Quase tudo são funções sobre estruturas simples — listas, tuplas, dicionários.
>
> Isso é uma escolha do autor, e ela combina com o propósito: uma função com anotação de tipo mostra o que entra e o que sai numa linha. Um método esconde metade disso no estado do objeto.

### Iteráveis e geradores

Aqui vale desacelerar de novo.

Uma lista de um milhão de elementos ocupa memória para um milhão de elementos, mesmo que você vá usar só os três primeiros. Um **gerador** produz os valores sob demanda.

Cria-se um com `yield`:

In [ ]:
def generate_range(n):
    i = 0
    while i < n:
        yield i     # cada chamada a yield produz um valor
        i += 1

for i in generate_range(10):
    print(i, end=" ")

A diferença aparece quando a sequência é infinita:

In [ ]:
def natural_numbers():
    """Devolve 1, 2, 3, ..."""
    n = 1
    while True:
        yield n
        n += 1

Essa função nunca termina — e ainda assim é útil, porque nada é calculado até alguém pedir.

O segundo jeito de criar geradores é uma compreensão com parênteses em vez de colchetes:

In [ ]:
data = natural_numbers()
evens = (x for x in data if x % 2 == 0)
even_squares = (x ** 2 for x in evens)
even_squares_ending_in_six = (x for x in even_squares if x % 10 == 6)

# nada foi calculado ainda. Agora sim:
[next(even_squares_ending_in_six) for _ in range(3)]

> **🔷 Conceito**
>
> Repare no que acabou de acontecer. Encadeamos quatro transformações sobre uma sequência **infinita** e pedimos três valores. Só os cálculos necessários para produzir esses três aconteceram.
>
> Se `natural_numbers()` fosse uma lista, o programa teria travado na primeira linha.
>
> O preço: um gerador só pode ser percorrido **uma vez**. Se você precisa passar duas vezes pelos mesmos dados, precisa de uma lista — ou de um gerador novo.

Por fim, quando você precisa do índice junto do valor, `enumerate` evita o contador manual:

In [ ]:
names = ["Alice", "Bob", "Charlie", "Debbie"]

for i, name in enumerate(names):
    print(f"nome {i} é {name}")

`enumerate` aparece o tempo todo no código do livro, sempre no lugar de um `i = 0` seguido de `i += 1`.

## Ferramentas e Anotações de Tipo

> **📌 Nota**
>
> Esta seção corresponde a *Randomness*, *Regular Expressions*, *Functional Programming*, *zip and Argument Unpacking*, *args and kwargs*, *Type Annotations* e *Welcome to DataSciencester!*, do capítulo 2 de Grus (2019).

> **💡 Dica — O que esta seção assume**
>
> Que você conhece aleatoriedade pseudoaleatória, expressões regulares e funções de alta ordem. **O que vale desacelerar: `zip` e, sobretudo, anotações de tipo** — que atravessam a maior parte das funções do livro e são a contribuição mais distintiva do Grus (2019).

### Aleatoriedade e sementes

O módulo `random` produz números pseudoaleatórios:

In [ ]:
import random

random.seed(10)   # fixa a semente

quatro_uniformes = [random.random() for _ in range(4)]
quatro_uniformes

Outras funções úteis: `randrange` para inteiros, `shuffle` para embaralhar no lugar, `choice` para sortear um elemento, e `sample` para sortear vários **sem** reposição.

In [ ]:
random.seed(10)

print("randrange(10):", random.randrange(10))
print("sample de 6 em 60:", random.sample(range(60), 6))

up_to_ten = list(range(1, 11))
random.shuffle(up_to_ten)
print("shuffle:", up_to_ten)

# com reposição: o mesmo elemento pode sair duas vezes
print("com reposição:", [random.choice(range(10)) for _ in range(4)])

> **❗ Importante — A semente não é opcional**
>
> `random.seed(10)` faz o gerador produzir sempre a mesma sequência. Sem ela, cada execução dá números diferentes — e um experimento cuja resposta muda a cada execução não é reproduzível. Quando o resultado mudar, você não vai saber se mudou porque o algoritmo mudou ou porque o sorteio saiu diferente; e outra pessoa, rodando o mesmo código, não consegue conferir o seu número.
>
> Por isso, todo trecho de código que amostra, embaralha ou inicializa pesos ao acaso começa fixando a semente. Você vai ver `random.seed(...)` em todos os capítulos que fazem isso. Repare que é o `random` da biblioteca padrão, não o `numpy.random` — este livro é Python puro.

### Expressões regulares, rapidamente

In [ ]:
import re

re_examples = [
    not re.match("a", "cat"),              # 'cat' não COMEÇA com 'a'
    re.search("a", "cat"),                 # mas 'cat' CONTÉM 'a'
    not re.search("c", "dog"),             # 'dog' não contém 'c'
    3 == len(re.split("[ab]", "carbs")),   # divide em a ou b: ['c','r','s']
    "R-D-" == re.sub("[0-9]", "-", "R2D2") # troca dígitos por hífens
]

assert all(re_examples), "todos os exemplos de regex deveriam ser True"
all(re_examples)

A distinção que mais confunde é `match` contra `search`: o primeiro ancora no início da string, o segundo procura em qualquer posição.

### `zip` e desempacotamento

Aqui vale desacelerar. `zip` combina iteráveis, elemento a elemento:

In [ ]:
list1 = ['a', 'b', 'c']
list2 = [1, 2, 3]

list(zip(list1, list2))

`zip` é **preguiçoso**: ele não produz nada até alguém iterar. É por isso que a linha acima precisa do `list()`.

E dá para desfazer o zip com o operador `*`, que desempacota uma lista em argumentos posicionais:

In [ ]:
pairs = [('a', 1), ('b', 2), ('c', 3)]
letters, numbers = zip(*pairs)

letters, numbers

> **🟩 Exemplo**
>
> `zip` é o motor da aritmética de vetores do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html):
>
> ```python
> def add(v: Vector, w: Vector) -> Vector:
>     return [v_i + w_i for v_i, w_i in zip(v, w)]
>
> def dot(v: Vector, w: Vector) -> float:
>     return sum(v_i * w_i for v_i, w_i in zip(v, w))
> ```
>
> "Percorrer duas sequências em paralelo" é a operação mais frequente em código numérico, e `zip` é como se escreve isso em Python. Sem ele, seria um laço por índice com `range(len(v))` — e aí você teria que lembrar de verificar se os comprimentos batem, coisa que o `assert` na primeira linha da função faz de propósito.

O `*` funciona em qualquer chamada de função, e `**` faz o mesmo com dicionários e argumentos nomeados:

In [ ]:
def other_way_magic(x, y, z):
    return x + y + z

x_y_list = [1, 2]
z_dict = {"z": 3}

assert other_way_magic(*x_y_list, **z_dict) == 6, "1 + 2 + 3 deveria dar 6"
other_way_magic(*x_y_list, **z_dict)

> **📌 Nota**
>
> Na definição de uma função, `*args` recolhe os argumentos posicionais numa tupla e `**kwargs` recolhe os nomeados num dicionário. É útil para escrever funções de alta ordem que aceitam qualquer assinatura.
>
> O livro usa isso com parcimônia, e o autor explica por quê: **o código fica mais correto e mais legível quando você é explícito sobre que argumentos a função exige.** `*args` e `**kwargs` só quando não há alternativa.

### Anotações de tipo

Esta é a parte mais importante da seção, e a mais distintiva do livro.

Python é **dinamicamente tipado**: em geral, ele não se importa com os tipos dos objetos, desde que sejam usados de forma válida.

In [ ]:
def add(a, b):
    return a + b

assert add(10, 5) == 15,              "+ é válido para números"
assert add([1, 2], [3]) == [1, 2, 3], "+ é válido para listas"
assert add("oi ", "lá") == "oi lá",   "+ é válido para strings"

try:
    add(10, "cinco")
except TypeError:
    print("não dá para somar um int com uma string")

Numa linguagem estaticamente tipada, funções e objetos teriam tipos específicos. Python permite escrever isso:

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

add(10, 5)

> **⚠️ Atenção — As anotações não fazem nada**
>
> Este é o ponto que mais confunde quem chega de uma linguagem estática.
>
> A função anotada acima **continua** somando strings sem reclamar, e `add(10, "cinco")` levanta exatamente o mesmo `TypeError` de antes. O interpretador não verifica coisa alguma.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

add("mesmo ", "assim")   # a anotação diz int; o Python não se importa

Então por que usá-las? O Grus (2019) dá quatro razões, e a primeira é a que mais importa aqui:

**1. Tipos são documentação.** Isso vale em dobro num livro que usa código para ensinar conceitos matemáticos. Compare:

```python
def dot_product(x, y): ...

def dot_product(x: Vector, y: Vector) -> float: ...
```

A segunda versão diz o que a função faz antes de você ler o corpo dela. Duas listas de números entram; um número sai. O autor escreve que se acostumou tanto com isso que hoje acha Python sem anotação difícil de ler.

**2. Ferramentas externas verificam.** O `mypy` lê o código, inspeciona as anotações e acusa erros de tipo **antes de você executar**. Como o `assert`, é uma forma de achar erro cedo. O autor conta que roda o mypy sobre todo o código do livro — é parte do que garante que ele esteja correto.

**3. Pensar nos tipos força um projeto melhor.** Uma função cujo parâmetro precisa ser anotado como `Union[str, int, float, bool]` está dizendo alguma coisa sobre si mesma: provavelmente é frágil e difícil de usar. A anotação torna o problema visível.

**4. O editor ajuda.** Com os tipos declarados, o editor autocompleta corretamente e reclama de erros na hora.

### Como escrever anotações

Para tipos embutidos, use o próprio tipo. Para coleções, o módulo `typing` traz versões parametrizadas — e é isso que dá a informação útil:

In [ ]:
from typing import List, Dict, Optional, Tuple, Callable

def total(xs: List[float]) -> float:      # lista de floats, não lista qualquer
    return sum(xs)

# quando o tipo não é óbvio pela atribuição, anote a variável
values: List[int] = []
best_so_far: Optional[float] = None       # pode ser float ou None

counts: Dict[str, int] = {'data': 1, 'science': 2}
triple: Tuple[int, float, int] = (10, 2.3, 5)

total([1.0, 2.0, 3.0])

Como Python tem funções de primeira classe, existe um tipo para representá-las:

In [ ]:
def twice(repeater: Callable[[str, int], str], s: str) -> str:
    """repeater é uma função que recebe (str, int) e devolve str"""
    return repeater(s, 2)

def comma_repeater(s: str, n: int) -> str:
    return ', '.join([s for _ in range(n)])

assert twice(comma_repeater, "dicas de tipo") == "dicas de tipo, dicas de tipo"
twice(comma_repeater, "dicas de tipo")

E, como anotações são apenas objetos Python, dá para dar nome a elas:

In [ ]:
Number = int
Numbers = List[Number]

def total(xs: Numbers) -> Number:
    return sum(xs)

total([1, 2, 3])

> **🔷 Conceito**
>
> Esse último recurso é o que produz o `Vector = List[float]` do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html) — o apelido que dá nome à ideia sem criar tipo novo.
>
> É um bom resumo do que as anotações fazem neste livro: elas não mudam o que o código executa. Mudam o que ele **diz**.

### Bem-vindo à DataSciencester!

Isto conclui a integração de novos funcionários. Ah, e mais uma coisa: tente não desviar dinheiro.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 2 de Grus (2019) sugere:

- O [tutorial oficial de Python](https://docs.python.org/pt-br/3/tutorial/), disponível em português, para quem quiser a cobertura completa da linguagem, do zero.
- O [tutorial oficial do IPython](https://ipython.readthedocs.io/en/stable/interactive/tutorial.html), se você decidir usá-lo — e o autor recomenda que use.
- A [documentação do mypy](https://mypy.readthedocs.io/), que conta mais do que você jamais quis saber sobre anotações e verificação de tipos em Python.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.